In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/raw")

files = sorted(DATA_DIR.glob("*.csv"))

data = {}

for file in files:
    data[file.stem] = pd.read_csv(file)

print(f"Loaded {len(data)} datasets")

Loaded 18 datasets


In [9]:
borrowers = data["borrowers"]
accounts = data["accounts"]
agents = data["agents"]
agent_sessions = data["agent_sessions"]
campaigns = data["campaigns"]
targeting = data["daily_targeting"]
calls = data["calls"]
attempts = data["call_attempts"]
dispositions = data["call_dispositions"]
whatsapp = data["whatsapp_events"]
sms = data["sms_events"]
field = data["field_visits"]
ptp = data["promises_to_pay"]
payments = data["payments"]
vendors = data["vendor_telephony"]
complaints = data["complaints"]
status_history = data["account_status_history"]

In [3]:
payment_id_counts = (
    payments["payment_id"]
    .value_counts()
)

payment_id_counts[
    payment_id_counts > 1
]

payment_id
PAYMENT0011759    2
PAYMENT0000794    2
PAYMENT0007582    2
PAYMENT0009866    2
PAYMENT0002242    2
                 ..
PAYMENT0024817    2
PAYMENT0024820    2
PAYMENT0024822    2
PAYMENT0024832    2
PAYMENT0024852    2
Name: count, Length: 500, dtype: int64

In [4]:
print(
    "Duplicate payment IDs:",
    (payment_id_counts > 1).sum()
)

Duplicate payment IDs: 500


In [5]:
ref_counts = (
    payments
    .dropna(subset=["payment_reference"])
    ["payment_reference"]
    .value_counts()
)

duplicate_refs = ref_counts[ref_counts > 1]

print("Duplicate payment references:", len(duplicate_refs))

display(duplicate_refs.head(20))

Duplicate payment references: 3745


payment_reference
TXN0000050468    5
TXN0000021482    5
TXN0000065723    5
TXN0000002005    5
TXN0000006936    5
TXN0000044312    5
TXN0000052559    5
TXN0000062318    4
TXN0000011850    4
TXN0000047758    4
TXN0000057961    4
TXN0000000512    4
TXN0000000776    4
TXN0000009490    4
TXN0000037089    4
TXN0000018649    4
TXN0000059566    4
TXN0000016608    4
TXN0000059686    4
TXN0000039096    4
Name: count, dtype: int64

In [6]:
duplicate_ref_values = duplicate_refs.index

duplicate_payment_records = payments[
    payments["payment_reference"].isin(
        duplicate_ref_values
    )
].sort_values("payment_reference")

display(duplicate_payment_records)

,payment_id,account_id,borrower_id,event_at,payment_reference,amount,payment_status,payment_method,provider_id
25010,PAYMENT0000552,ACC0021942,BRW0011343,2026-08-06 18:56:06,TXN0000000009,11792.14,SUCCESS,UPI,VND0000012
551,PAYMENT0000552,ACC0021942,BRW0011343,2026-08-06 18:56:06,TXN0000000009,11792.14,SUCCESS,UPI,VND0000012
1387,PAYMENT0001388,ACC0019483,BRW0003324,2026-01-31 05:00:49,TXN0000000027,53610.92,FAILED,CARD,VND0000005
3477,PAYMENT0003478,ACC0022277,BRW0011422,2026-06-09 12:22:41,TXN0000000027,43010.39,SUCCESS,NACH,VND0000010
5306,PAYMENT0005307,ACC0003975,BRW0007408,2026-05-24 14:49:16,TXN0000000032,18086.24,PENDING,CARD,VND0000003
...,...,...,...,...,...,...,...,...,...
2423,PAYMENT0002424,ACC0006446,BRW0001634,2026-05-13 13:07:00,TXN0000069980,125274.77,SUCCESS,CARD,VND0000009
10136,PAYMENT0010137,ACC0014203,BRW0002421,2026-03-15 12:59:22,TXN0000069980,127665.67,SUCCESS,NETBANKING,VND0000013
16050,PAYMENT0016051,ACC0017573,BRW0003657,2026-01-30 12:17:17,TXN0000069984,79112.73,SUCCESS,NETBANKING,VND0000013
25297,PAYMENT0016051,ACC0017573,BRW0003657,2026-01-30 12:17:17,TXN0000069984,79112.73,SUCCESS,NETBANKING,VND0000013


In [10]:
payments["payment_status"].value_counts(dropna=False)
successful = payments[
    payments["payment_status"].str.upper().isin(
        ["SUCCESS", "SUCCESSFUL", "COMPLETED", "PAID"]
    )
].copy()
successful_ref_counts = (
    successful
    .dropna(subset=["payment_reference"])
    ["payment_reference"]
    .value_counts()
)

duplicate_success_refs = successful_ref_counts[
    successful_ref_counts > 1
]

print(
    "Duplicate successful payment references:",
    len(duplicate_success_refs)
)
duplicate_success_payments = successful[
    successful["payment_reference"].isin(
        duplicate_success_refs.index
    )
]

print(
    "Rows involved:",
    len(duplicate_success_payments)
)

print(
    "Amount involved: ₹",
    duplicate_success_payments["amount"].sum()
)

Duplicate successful payment references: 2033
Rows involved: 4299
Amount involved: ₹ 325954259.77


In [11]:
campaigns["start_at"] = pd.to_datetime(
    campaigns["start_at"],
    errors="coerce"
)

campaigns["end_at"] = pd.to_datetime(
    campaigns["end_at"],
    errors="coerce"
)

display(
    campaigns[
        [
            "campaign_id",
            "campaign_name",
            "channel",
            "strategy_version",
            "target_definition",
            "start_at",
            "end_at"
        ]
    ].head(20)
)

,campaign_id,campaign_name,channel,strategy_version,target_definition,start_at,end_at
0,CMP0000001,DIGITAL_FOLLOWUP,FIELD,legacy,DPD>=30,2026-02-17 06:56:01,2026-04-24 06:56:01
1,CMP0000002,BOUNCE,MIXED,v2,DPD>=60,2026-04-30 17:51:31,2026-05-18 17:51:31
2,CMP0000003,30DPD_W1,MIXED,v1,DPD>=30,2026-04-11 16:02:51,2026-05-18 16:02:51
3,CMP0000004,BOUNCE,VOICE,legacy,DPD>=60,2026-04-28 06:15:30,2026-05-29 06:15:30
4,CMP0000005,60DPD_INTENT,WHATSAPP,legacy,DPD>=60,2026-05-04 04:16:21,2026-07-08 04:16:21
5,CMP0000006,BOUNCE,MIXED,legacy,DPD>=60,2026-01-23 09:30:10,2026-04-13 09:30:10
6,CMP0000007,DIGITAL_FOLLOWUP,MIXED,v2,PROMISE_BROKEN,2026-02-14 15:20:05,2026-04-22 15:20:05
7,CMP0000008,NPA_RECOVERY,SMS,v2,PROMISE_BROKEN,2026-05-21 10:58:25,2026-06-27 10:58:25
8,CMP0000009,60DPD_INTENT,WHATSAPP,v1,HIGH_RISK,2026-04-18 15:47:29,2026-06-07 15:47:29
9,CMP0000010,30DPD_W1,FIELD,v1,DPD>=30,2026-02-14 13:09:24,2026-04-30 13:09:24


In [12]:
targeting["target_date"] = pd.to_datetime(
    targeting["target_date"],
    errors="coerce"
)

target_campaign = targeting.merge(
    campaigns[
        [
            "campaign_id",
            "start_at",
            "end_at"
        ]
    ],
    on="campaign_id",
    how="left"
)

target_campaign["within_campaign"] = (
    (target_campaign["target_date"] >= target_campaign["start_at"].dt.normalize()) &
    (target_campaign["target_date"] <= target_campaign["end_at"].dt.normalize())
)

print(
    "Targeting records outside campaign dates:",
    (~target_campaign["within_campaign"]).sum()
)

Targeting records outside campaign dates: 36139


In [13]:
payments["event_at"] = pd.to_datetime(
    payments["event_at"],
    errors="coerce"
)

calls["event_at"] = pd.to_datetime(
    calls["event_at"],
    errors="coerce"
)

ptp["event_at"] = pd.to_datetime(
    ptp["event_at"],
    errors="coerce"
)

In [14]:
print(
    "Payments:",
    payments["event_at"].min(),
    "→",
    payments["event_at"].max()
)

print(
    "Calls:",
    calls["event_at"].min(),
    "→",
    calls["event_at"].max()
)

Payments: 2026-01-01 00:14:40 → 2026-08-08 23:50:23
Calls: 2025-12-29 06:52:37 → 2026-08-12 15:43:05


In [15]:
for name, df in data.items():

    if "timezone" in df.columns:

        print("\n", name)

        print(
            df["timezone"]
            .value_counts(dropna=False)
        )


 accounts
timezone
UTC             10096
Asia/Kolkata     9981
Asia/Dubai       9923
Name: count, dtype: int64

 agent_sessions
timezone
Asia/Kolkata    7506
UTC             7494
Name: count, dtype: int64

 calls
timezone
Asia/Kolkata    30485
Asia/Dubai      30464
UTC             30401
Name: count, dtype: int64

 vendor_telephony
timezone
UTC             8
Asia/Kolkata    7
Name: count, dtype: int64


In [16]:
calls["timezone"].value_counts(dropna=False)

timezone
Asia/Kolkata    30485
Asia/Dubai      30464
UTC             30401
Name: count, dtype: int64

In [17]:
agent_sessions["timezone"].value_counts(dropna=False)

timezone
Asia/Kolkata    7506
UTC             7494
Name: count, dtype: int64

In [18]:
accounts["timezone"].value_counts(dropna=False)

timezone
UTC             10096
Asia/Kolkata     9981
Asia/Dubai       9923
Name: count, dtype: int64

In [19]:
calls["event_at_parsed"] = pd.to_datetime(
    calls["event_at"],
    errors="coerce"
)

calls["hour"] = calls["event_at_parsed"].dt.hour

display(
    calls["hour"]
    .value_counts()
    .sort_index()
)

hour
0     3688
1     3799
2     3942
3     3831
4     3751
5     3864
6     3871
7     3850
8     3765
9     3751
10    3854
11    3891
12    3716
13    3871
14    3767
15    3685
16    3903
17    3683
18    3844
19    3841
20    3740
21    3767
22    3727
23    3949
Name: count, dtype: int64

In [20]:
display(
    vendors[
        [
            "vendor_id",
            "vendor_name",
            "vendor_account_id",
            "timezone",
            "status",
            "schema_version"
        ]
    ]
)

,vendor_id,vendor_name,vendor_account_id,timezone,status,schema_version
0,VND0000001,Airtel,VAC342762,Asia/Kolkata,INACTIVE,v3
1,VND0000002,Exotel,VAC456766,UTC,INACTIVE,v3
2,VND0000003,Twilio,VAC074211,UTC,INACTIVE,v3
3,VND0000004,Twilio,VAC321507,UTC,ACTIVE,v1
4,VND0000005,Twilio,VAC976733,Asia/Kolkata,ACTIVE,v1
5,VND0000006,Knowlarity,VAC544470,UTC,ACTIVE,v2
6,VND0000007,TataTele,VAC997592,UTC,INACTIVE,v1
7,VND0000008,Airtel,VAC496672,Asia/Kolkata,INACTIVE,v3
8,VND0000009,Exotel,VAC382568,Asia/Kolkata,INACTIVE,v1
9,VND0000010,Airtel,VAC172916,UTC,INACTIVE,v1


In [21]:
vendors[
    ["vendor_name", "schema_version"]
].value_counts()

vendor_name  schema_version
Airtel       v3                3
             v1                2
Twilio       v1                2
Exotel       v1                1
             v3                1
Knowlarity   v2                1
             v1                1
             v3                1
TataTele     v1                1
             v2                1
Twilio       v3                1
Name: count, dtype: int64

In [22]:
calls["month"] = calls["event_at"].dt.to_period("M").astype(str)

vendor_month = (
    calls
    .groupby(["month", "vendor_id"])
    .size()
    .reset_index(name="calls")
)

display(
    vendor_month.sort_values(
        ["month", "calls"],
        ascending=[True, False]
    )
)

,month,vendor_id,calls
0,2025-12,VND0000013,1
10,2026-01,VND0000010,906
7,2026-01,VND0000007,892
1,2026-01,VND0000001,889
3,2026-01,VND0000003,885
...,...,...,...
118,2026-08,VND0000013,211
117,2026-08,VND0000012,203
107,2026-08,VND0000002,202
120,2026-08,VND0000015,201


In [23]:
vendor_status = pd.crosstab(
    calls["vendor_id"],
    calls["call_status"],
    normalize="index"
) * 100

display(vendor_status.round(2))

call_status,ANSWERED,BUSY,FAILED,NO_ANSWER,VOICEMAIL
vendor_id,,,,,
VND0000001,19.34,19.96,20.04,20.30,20.37
VND0000002,19.65,19.05,20.39,20.61,20.30
VND0000003,19.92,19.41,20.34,20.02,20.30
VND0000004,19.51,21.12,20.47,18.71,20.19
VND0000005,19.52,19.96,19.84,20.77,19.91
VND0000006,20.89,19.75,19.89,20.29,19.19
VND0000007,19.44,19.93,19.60,20.46,20.58
VND0000008,19.77,19.64,20.27,19.65,20.67
VND0000009,19.25,20.44,20.59,20.10,19.62


In [24]:
agent_mapping = (
    agents
    .groupby("agent_id")["employee_code"]
    .nunique()
    .sort_values(ascending=False)
)

display(
    agent_mapping.head(20)
)

agent_id
AGT0000367    48
AGT0000533    47
AGT0000875    47
AGT0000843    46
AGT0000876    45
AGT0000563    45
AGT0000540    45
AGT0000543    45
AGT0000056    44
AGT0000440    44
AGT0000496    43
AGT0000883    43
AGT0000531    43
AGT0000746    43
AGT0000975    43
AGT0000280    42
AGT0000992    42
AGT0000544    42
AGT0000435    42
AGT0000649    41
Name: employee_code, dtype: int64

In [25]:
multi_employee_agents = agent_mapping[
    agent_mapping > 1
]

print(
    "Agents with multiple employee codes:",
    len(multi_employee_agents)
)

Agents with multiple employee codes: 1000


In [26]:
employee_mapping = (
    agents
    .groupby("employee_code")["agent_id"]
    .nunique()
    .sort_values(ascending=False)
)

display(
    employee_mapping.head(20)
)

employee_code
EMP00900    46
EMP00917    43
EMP00749    43
EMP00195    42
EMP00288    42
EMP00661    42
EMP00351    41
EMP00753    40
EMP00196    40
EMP00171    40
EMP00837    40
EMP00454    40
EMP00676    39
EMP00679    39
EMP00139    39
EMP00182    38
EMP00538    38
EMP00283    38
EMP00340    38
EMP00346    38
Name: agent_id, dtype: int64

In [27]:
multi_id_employees = employee_mapping[
    employee_mapping > 1
]

print(
    "Employee codes linked to multiple agent IDs:",
    len(multi_id_employees)
)

Employee codes linked to multiple agent IDs: 1099


In [28]:
if len(multi_employee_agents) > 0:

    agent_id = multi_employee_agents.index[0]

    display(
        agents[
            agents["agent_id"] == agent_id
        ].sort_values("updated_at")
    )

,agent_id,employee_code,agent_name,vendor_id,team,status,joined_at,updated_at
10403,AGT0000367,EMP00944,Pooja Nair,VND0000002,DIGITAL,INACTIVE,2025-09-12 10:03:42,2025-01-15 17:42:34
19672,AGT0000367,EMP00735,Ananya Rao,VND0000005,T2,ACTIVE,2025-02-16 18:32:41,2025-01-27 06:04:41
21329,AGT0000367,EMP00141,Rohan Patel,VND0000008,T1,INACTIVE,2024-02-19 03:20:29,2025-01-29 07:39:27
1961,AGT0000367,EMP00034,Ananya Rao,VND0000012,DIGITAL,SUSPENDED,2025-02-08 12:11:57,2025-02-23 02:02:20
8092,AGT0000367,EMP00407,Amit Kumar,VND0000008,T3,ACTIVE,2025-04-11 10:28:58,2025-03-01 10:25:35
20491,AGT0000367,EMP00859,Ananya Rao,VND0000010,FIELD,SUSPENDED,2024-10-19 06:35:35,2025-03-10 23:40:41
6955,AGT0000367,EMP00816,Amit Kumar,VND0000014,T1,ACTIVE,2025-05-24 17:18:36,2025-04-23 21:04:54
4101,AGT0000367,EMP00285,Rahul Verma,VND0000009,DIGITAL,ACTIVE,2025-05-30 22:51:47,2025-04-25 12:58:08
11522,AGT0000367,EMP00326,Amit Kumar,VND0000011,T2,SUSPENDED,2024-10-06 16:53:51,2025-05-25 12:13:04
27477,AGT0000367,EMP00130,Sneha Das,VND0000007,FIELD,ACTIVE,2024-09-16 22:24:06,2025-05-30 17:45:18


In [29]:
for col in [
    "loan_type",
    "dpd",
    "risk_segment",
    "status",
    "timezone"
]:
    print("\n", col)
    print(
        accounts[col]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )


 loan_type
loan_type
CREDIT_CARD    20.27
AUTO           20.26
PERSONAL       19.94
CONSUMER       19.77
BNPL           19.76
Name: proportion, dtype: float64

 dpd
dpd
60     9.23
120    9.20
45     9.15
75     9.14
15     9.12
5      9.09
90     9.09
1      9.04
30     9.01
180    8.98
0      8.95
Name: proportion, dtype: float64

 risk_segment
risk_segment
HIGH      25.17
MEDIUM    25.11
LOW       25.04
NPA       24.67
Name: proportion, dtype: float64

 status
status
ACTIVE      25.13
CLOSED      24.99
PAID        24.95
WRITEOFF    24.93
Name: proportion, dtype: float64

 timezone
timezone
UTC             33.65
Asia/Kolkata    33.27
Asia/Dubai      33.08
Name: proportion, dtype: float64


In [30]:
accounts["opened_at"] = pd.to_datetime(
    accounts["opened_at"],
    errors="coerce"
)

accounts["opened_month"] = (
    accounts["opened_at"]
    .dt.to_period("M")
    .astype(str)
)

In [31]:
monthly_portfolio = (
    accounts
    .groupby(["opened_month", "risk_segment"])
    .size()
    .groupby(level=0)
    .transform(lambda x: x / x.sum() * 100)
    .reset_index(name="pct")
)

display(monthly_portfolio)

,opened_month,risk_segment,pct
0,2024-01,HIGH,24.180015
1,2024-01,LOW,26.086957
2,2024-01,MEDIUM,26.315789
3,2024-01,NPA,23.417239
4,2024-02,HIGH,24.172440
...,...,...,...
87,2025-10,NPA,25.433962
88,2025-11,HIGH,24.481659
89,2025-11,LOW,26.634769
90,2025-11,MEDIUM,24.561404


In [32]:
event_sources = []

for name, df in {
    "calls": calls,
    "payments": payments,
    "ptp": ptp,
    "whatsapp": whatsapp,
    "sms": sms,
    "field": field,
    "targeting": targeting
}.items():

    temp = df.copy()

    time_col = (
        "target_date"
        if name == "targeting"
        else "event_at"
    )

    temp[time_col] = pd.to_datetime(
        temp[time_col],
        errors="coerce"
    )

    temp["month"] = (
        temp[time_col]
        .dt.to_period("M")
        .astype(str)
    )

    temp["source"] = name

    event_sources.append(
        temp[["account_id", "month", "source"]]
    )

account_activity = pd.concat(
    event_sources,
    ignore_index=True
)

account_activity = (
    account_activity
    .drop_duplicates(
        ["account_id", "month", "source"]
    )
)

In [34]:
monthly_active_accounts = (
    account_activity
    .groupby("month")["account_id"]
    .nunique()
)

display(monthly_active_accounts)

month
2025-12        1
2026-01    22995
2026-02    21938
2026-03    22914
2026-04    22576
2026-05    22944
2026-06    22537
2026-07    22978
2026-08     9296
Name: account_id, dtype: int64

In [35]:
monthly_account_sets = {
    month: set(group["account_id"])
    for month, group in account_activity.groupby("month")
}

months = sorted(monthly_account_sets)

for i in range(1, len(months)):

    previous = monthly_account_sets[months[i - 1]]
    current = monthly_account_sets[months[i]]

    disappeared = previous - current

    print(
        months[i - 1],
        "→",
        months[i],
        "| disappeared:",
        len(disappeared)
    )

2025-12 → 2026-01 | disappeared: 1
2026-01 → 2026-02 | disappeared: 6230
2026-02 → 2026-03 | disappeared: 5169
2026-03 → 2026-04 | disappeared: 5658
2026-04 → 2026-05 | disappeared: 5279
2026-05 → 2026-06 | disappeared: 5739
2026-06 → 2026-07 | disappeared: 5265
2026-07 → 2026-08 | disappeared: 15920


In [36]:
status_history["event_at"] = pd.to_datetime(
    status_history["event_at"],
    errors="coerce"
)

status_history["month"] = (
    status_history["event_at"]
    .dt.to_period("M")
    .astype(str)
)

display(
    status_history["status"]
    .value_counts()
)

status
PAID          8650
CLOSED        8614
DELINQUENT    8612
NPA           8612
WRITEOFF      8583
ACTIVE        8518
PTP           8411
Name: count, dtype: int64

In [38]:
findings = pd.DataFrame(columns=[
    "issue",
    "dataset",
    "detection_method",
    "observation",
    "business_impact",
    "classification",
    "action"
])

display(findings)

,issue,dataset,detection_method,observation,business_impact,classification,action
